In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time


def scrape_job_data(base_url, total_pages=3):
    """Scrapes job data from a mock scrapable site across multiple pages."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    all_jobs = []

    print("--- Starting Web Scraping Task ---")

    for page in range(1, total_pages + 1):
        # Using a reliable, stable static page meant for scraping practice
        url = f"{base_url}?page={page}"
        print(f"Scraping page {page}...")

        try:
            response = requests.get(url, headers=headers, timeout=10)

            # Check if the page loaded successfully
            if response.status_code != 200:
                print(
                    f"Skipping page {page}: Status code {response.status_code}"
                )
                continue

            soup = BeautifulSoup(response.content, "html.parser")

            # Target the container for each job post (Matches standard sandbox structures)
            job_cards = soup.find_all("div", class_="card-content")

            if not job_cards:
                # Fallback for alternative common sandbox structures (like RealPython fake-jobs)
                job_cards = soup.find_all("div", class_="box")

            for card in job_cards:
                title_element = card.find("h2", class_="title")
                company_element = card.find("h3", class_="company")
                location_element = card.find("p", class_="location")
                date_element = card.find("time")

                # Extract text safely to avoid NoneType attribute errors
                title = title_element.text.strip() if title_element else "N/A"
                company = (
                    company_element.text.strip() if company_element else "N/A"
                )
                location = (
                    location_element.text.strip() if location_element else "N/A"
                )
                date_posted = (
                    date_element.text.strip() if date_element else "N/A"
                )

                all_jobs.append(
                    {
                        "Job Title": title,
                        "Company": company,
                        "Location": location,
                        "Date Posted": date_posted,
                    }
                )

            # Polite scraping practice: pause between requests
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"An error occurred while fetching page {page}: {e}")
            break

    # Convert to DataFrame and save
    if all_jobs:
        df = pd.DataFrame(all_jobs)
        output_file = "scraped_job_data.csv"
        df.to_csv(output_file, index=False)
        print(f"\n Success! Successfully scraped {len(df)} jobs.")
        print(f"Data saved to environment as '{output_file}'")
        return df
    else:
        print("\n No data was scraped. Check your target URL structure.")
        return None


# Execution Example (Using standard public scraping playground URL)
TARGET_URL = "https://realpython.github.io/fake-jobs/"
job_dataframe = scrape_job_data(TARGET_URL, total_pages=1)
if job_dataframe is not None:
    print(job_dataframe.head())
    

--- Starting Web Scraping Task ---
Scraping page 1...

 Success! Successfully scraped 100 jobs.
Data saved to environment as 'scraped_job_data.csv'
                 Job Title                     Company              Location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

  Date Posted  
0  2021-04-08  
1  2021-04-08  
2  2021-04-08  
3  2021-04-08  
4  2021-04-08  


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time


class JobScraper:
    """
    A professional web scraper for extracting job listings
    from Real Python's Fake Jobs website.
    """

    def __init__(self, url):
        self.url = url
        self.headers = {
            "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0 Safari/537.36"
        }

    def fetch_page(self):
        """Fetch webpage content."""
        try:
            response = requests.get(
                self.url,
                headers=self.headers,
                timeout=10
            )

            response.raise_for_status()

            print("✅ Website fetched successfully.")
            return response.text

        except requests.exceptions.RequestException as e:
            print(f"❌ Error fetching webpage: {e}")
            return None

    def parse_jobs(self, html):
        """Parse HTML and extract job details."""

        soup = BeautifulSoup(html, "html.parser")

        job_cards = soup.find_all("div", class_="card-content")

        jobs = []

        for card in job_cards:

            title = card.find("h2", class_="title")
            company = card.find("h3", class_="company")
            location = card.find("p", class_="location")
            date = card.find("time")

            jobs.append({
                "Job Title":
                    title.text.strip() if title else "N/A",

                "Company":
                    company.text.strip() if company else "N/A",

                "Location":
                    location.text.strip() if location else "N/A",

                "Date Posted":
                    date["datetime"] if date else "N/A"
            })

        return jobs

    def save_data(self, jobs):
        """Save data to CSV."""

        df = pd.DataFrame(jobs)

        # Remove duplicates
        df.drop_duplicates(inplace=True)

        # Reset index
        df.reset_index(drop=True, inplace=True)

        file_name = "scraped_job_data.csv"

        df.to_csv(file_name, index=False)

        print(f"\n✅ Dataset saved as '{file_name}'")
        print(f"✅ Total Jobs Scraped: {len(df)}")

        return df

    def display_summary(self, df):
        """Display dataset summary."""

        print("\n" + "=" * 50)
        print("DATASET SUMMARY")
        print("=" * 50)

        print(f"\nTotal Records: {len(df)}")
        print(f"Total Columns: {len(df.columns)}")

        print("\nColumn Names:")
        print(df.columns.tolist())

        print("\nMissing Values:")
        print(df.isnull().sum())

        print("\nFirst 5 Records:")
        print(df.head())

    def run(self):
        """Main execution method."""

        print("=" * 50)
        print("CODEALPHA TASK 1 - WEB SCRAPING PROJECT")
        print("=" * 50)

        html = self.fetch_page()

        if html:

            jobs = self.parse_jobs(html)

            if jobs:

                df = self.save_data(jobs)

                self.display_summary(df)

                print("\n🎉 Web Scraping Completed Successfully!")

                return df

            else:
                print("❌ No job data found.")

        return None


# ==========================
# MAIN PROGRAM
# ==========================

if __name__ == "__main__":

    TARGET_URL = "https://realpython.github.io/fake-jobs/"

    scraper = JobScraper(TARGET_URL)

    job_df = scraper.run()

CODEALPHA TASK 1 - WEB SCRAPING PROJECT
✅ Website fetched successfully.

✅ Dataset saved as 'scraped_job_data.csv'
✅ Total Jobs Scraped: 100

DATASET SUMMARY

Total Records: 100
Total Columns: 4

Column Names:
['Job Title', 'Company', 'Location', 'Date Posted']

Missing Values:
Job Title      0
Company        0
Location       0
Date Posted    0
dtype: int64

First 5 Records:
                 Job Title                     Company              Location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

  Date Posted  
0  2021-04-08  
1  2021-04-08  
2  2021-04-08  
3  2021-04-08  
4  2021-04-08  

🎉 Web Scraping Completed Successfully!
